In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Load dataset
df = pd.read_csv('synthetic_travel_recommendation_dataset.csv')
print(f"Original Dataset Shape: {df.shape}")
print("\n" + "="*50)


Original Dataset Shape: (20000, 26)



In [1]:
import ipywidgets as widgets
widgets.IntSlider()
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Churn EDA Report", explorative=True)
profile.to_file("eda_report.html") # HTML report banega
profile.to_notebook_iframe() # Direct notebook mein dekho

ModuleNotFoundError: No module named 'ipywidgets'

In [3]:
# 1. MISSING VALUE ANALYSIS
print("\n📊 MISSING VALUES ANALYSIS:")
print(df.isnull().sum())




📊 MISSING VALUES ANALYSIS:
place_id                    0
place_name                  0
state                       0
latitude                    0
longitude                   0
place_type                  0
month                       0
climate                     0
popularity_level            0
accessibility               0
budget_thousand_inr         0
trip_duration_days          0
companions                  0
interest                    0
safety_index                0
entry_fee                   0
user_id                     0
session_id                  0
rating_given                0
review_text                 0
travel_dates_start          0
travel_dates_end            0
device_type                 0
engagement_score            0
mobility_restrictions    6640
language_preference         0
dtype: int64


In [4]:
# Handle missing values intelligently
if df.isnull().sum().sum() > 0:
    # Numerical columns: fill with median
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)
    
    # Categorical columns: fill with mode
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mode()[0], inplace=True)

print("\n✅ Missing values handled!")




✅ Missing values handled!


In [4]:
# 2. DUPLICATE REMOVAL
duplicates = df.duplicated().sum()
print(f"\n🔍 Duplicates Found: {duplicates}")
df.drop_duplicates(inplace=True)
print(f"✅ Dataset after removing duplicates: {df.shape}")



🔍 Duplicates Found: 0
✅ Dataset after removing duplicates: (20000, 26)


In [5]:
# 3. FEATURE ENGINEERING
print("\n🛠️ FEATURE ENGINEERING:")

# Convert date columns to datetime
df['travel_dates_start'] = pd.to_datetime(df['travel_dates_start'])
df['travel_dates_end'] = pd.to_datetime(df['travel_dates_end'])

# Extract useful features from dates
df['travel_month'] = df['travel_dates_start'].dt.month
df['travel_year'] = df['travel_dates_start'].dt.year
df['is_weekend'] = df['travel_dates_start'].dt.dayofweek.isin([5, 6]).astype(int)
df['season'] = df['travel_month'].apply(lambda x: 
    'Winter' if x in [12, 1, 2] else
    'Summer' if x in [3, 4, 5] else
    'Monsoon' if x in [6, 7, 8] else 'Autumn'
)

# Budget per day
df['budget_per_day'] = df['budget_thousand_inr'] / df['trip_duration_days']

# Create target variable: High rating = 1, Low rating = 0
df['high_rating'] = (df['rating_given'] >= 4.0).astype(int)

print("✅ New features created: travel_month, season, budget_per_day, high_rating")




🛠️ FEATURE ENGINEERING:
✅ New features created: travel_month, season, budget_per_day, high_rating


In [6]:
# 4. ENCODING CATEGORICAL VARIABLES
print("\n🔤 ENCODING CATEGORICAL VARIABLES:")

# Ordinal Encoding for variables with natural order
ordinal_features = {
    'accessibility': ['Easy', 'Moderate', 'Hard'],
    'popularity_level': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    'safety_index': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
}

# Ordinal encoding for accessibility (has natural order)
accessibility_mapping = {'Easy': 0, 'Moderate': 1, 'Hard': 2}
df['accessibility_encoded'] = df['accessibility'].map(accessibility_mapping)

# Label Encoding for nominal categorical variables
label_encoders = {}
nominal_cols = ['place_name', 'state', 'place_type', 'month', 'climate', 
                'companions', 'interest', 'device_type', 'mobility_restrictions', 
                'language_preference', 'season']

for col in nominal_cols:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"✅ Encoded: {col}")




🔤 ENCODING CATEGORICAL VARIABLES:
✅ Encoded: place_name
✅ Encoded: state
✅ Encoded: place_type
✅ Encoded: month
✅ Encoded: climate
✅ Encoded: companions
✅ Encoded: interest
✅ Encoded: device_type
✅ Encoded: mobility_restrictions
✅ Encoded: language_preference
✅ Encoded: season


In [7]:
# 5. FEATURE SELECTION FOR MODEL
print("\n🎯 SELECTING FEATURES FOR MODEL:")

feature_cols = [
    'latitude', 'longitude', 'place_type_encoded', 'month_encoded', 
    'climate_encoded', 'popularity_level', 'accessibility_encoded',
    'budget_thousand_inr', 'trip_duration_days', 'companions_encoded',
    'interest_encoded', 'safety_index', 'entry_fee', 'engagement_score',
    'mobility_restrictions_encoded', 'language_preference_encoded',
    'travel_month', 'is_weekend', 'season_encoded', 'budget_per_day'
]

target_col = 'high_rating'

X = df[feature_cols]
y = df[target_col]

print(f"✅ Features selected: {len(feature_cols)} features")
print(f"✅ Target variable: {target_col}")




🎯 SELECTING FEATURES FOR MODEL:
✅ Features selected: 20 features
✅ Target variable: high_rating


In [8]:
# 6. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 TRAIN-TEST SPLIT:")
print(f"Training Set: {X_train.shape}")
print(f"Testing Set: {X_test.shape}")
print(f"Class distribution in train: {y_train.value_counts().to_dict()}")




📊 TRAIN-TEST SPLIT:
Training Set: (16000, 20)
Testing Set: (4000, 20)
Class distribution in train: {0: 11837, 1: 4163}


In [9]:
# 7. FEATURE SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Feature scaling completed!")



✅ Feature scaling completed!


In [10]:
# 8. SAVE PREPROCESSED DATA AND ENCODERS
print("\n💾 SAVING PREPROCESSED DATA:")

# Create directory for saved files
import os
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Save train-test data
np.save('data/X_train_scaled.npy', X_train_scaled)
np.save('data/X_test_scaled.npy', X_test_scaled)
np.save('data/y_train.npy', y_train)
np.save('data/y_test.npy', y_test)

# Save feature names
with open('data/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

# Save encoders and scaler
with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save processed dataframe for EDA
df.to_csv('data/processed_dataset.csv', index=False)

print("✅ All files saved successfully!")
print("\n📁 Files saved:")
print("  - data/X_train_scaled.npy")
print("  - data/X_test_scaled.npy")
print("  - data/y_train.npy")
print("  - data/y_test.npy")
print("  - data/feature_names.pkl")
print("  - models/label_encoders.pkl")
print("  - models/scaler.pkl")
print("  - data/processed_dataset.csv")

print("\n" + "="*50)
print("🎉 PREPROCESSING COMPLETE! Ready for model training!")
print("="*50)


💾 SAVING PREPROCESSED DATA:
✅ All files saved successfully!

📁 Files saved:
  - data/X_train_scaled.npy
  - data/X_test_scaled.npy
  - data/y_train.npy
  - data/y_test.npy
  - data/feature_names.pkl
  - models/label_encoders.pkl
  - models/scaler.pkl
  - data/processed_dataset.csv

🎉 PREPROCESSING COMPLETE! Ready for model training!
